### Qlearn3Regime ground-truth recovery

Simulate synthetic data where each block's generating "strategy" is tied
to its true reward-probability tier (low_low / high_low / high_high, per
`mab_abstract_datagen.make_tiered_task`), using `Qlearn3Regime` itself
with a hard, externally-specified regime path (`DecisionModel.simulate_regime_path`)
instead of one driven by its own fitted transition matrix. Then fit
`Qlearn3Regime` blind (no block-type label) and check whether it recovers
the true regime assignment and parameters — a positive-control /
identifiability check before trusting anything the same model says about
real animal data.

In [ ]:
import numpy as np
from scipy.optimize import linear_sum_assignment
from banditpy.models.policy.regime import Qlearn3Regime
from banditpy.models.model import DecisionModel

# Real task tiers (mab_abstract_datagen.make_tiered_task): pool [0.2,0.3,0.4,0.6,0.7,0.8],
# n_high = count of arms >= 0.5; low_low: n_high==0, high_low: n_high==1, high_high: n_high==2
LOW_LOW = [(0.2, 0.3), (0.2, 0.4), (0.3, 0.4), (0.3, 0.2), (0.4, 0.2), (0.4, 0.3)]
HIGH_LOW = [
    (0.2, 0.8),
    (0.3, 0.7),
    (0.4, 0.6),
    (0.8, 0.2),
    (0.7, 0.3),
    (0.6, 0.4),
    (0.2, 0.6),
    (0.2, 0.7),
    (0.3, 0.6),
    (0.3, 0.8),
    (0.4, 0.7),
    (0.4, 0.8),
]
HIGH_HIGH = [(0.6, 0.7), (0.6, 0.8), (0.7, 0.8), (0.7, 0.6), (0.8, 0.6), (0.8, 0.7)]
TIER_POOLS = {0: LOW_LOW, 1: HIGH_LOW, 2: HIGH_HIGH}
TIER_NAMES = {0: "low_low", 1: "high_low", 2: "high_high"}

In [ ]:
policy_gen = Qlearn3Regime()
true_params = dict(
    alpha_c_1=0.2,
    alpha_u_1=0.0,  # agent 1 = low_low
    alpha_c_2=0.1,
    alpha_u_2=-0.1,  # agent 2 = high_low
    alpha_c_3=0.05,
    alpha_u_3=-0.2,  # agent 3 = high_high
    beta_q1_0=6.0,
    beta_q2_0=0.0,
    beta_q3_0=0.0,  # regime 0 -> agent 1
    beta_q1_1=0.0,
    beta_q2_1=6.0,
    beta_q3_1=0.0,  # regime 1 -> agent 2
    beta_q1_2=0.0,
    beta_q2_2=0.0,
    beta_q3_2=6.0,  # regime 2 -> agent 3
    beta_bias=0.0,
    stay_0=0.95,
    stay_1=0.95,
    stay_2=0.95,
)

n_blocks = 100
rng = np.random.default_rng(1)
block_regimes = rng.integers(0, 3, size=n_blocks).tolist()
block_probs = [TIER_POOLS[k][rng.integers(len(TIER_POOLS[k]))] for k in block_regimes]

task, true_regime = DecisionModel.simulate_regime_path(
    policy_gen, block_probs, block_regimes, 100, params=true_params, seed=2
)
print("total trials:", task.n_trials)
print("tier counts:", {TIER_NAMES[k]: block_regimes.count(k) for k in range(3)})

In [ ]:
model = DecisionModel(task, Qlearn3Regime(), reset_mode="session")
model.fit(n_starts=40, seed=0, optimizer="optuna", progress=False)
print("fitted params:", {k: round(v, 3) for k, v in model.params.items()})
print(
    "NLL:",
    model.nll,
    " per-trial:",
    model.nll / task.n_trials,
    " (chance =",
    np.log(2),
    ")",
)

In [ ]:
state_traj = model.get_state_trajectory()
pred_regime = state_traj.argmax(axis=1)

conf = np.zeros((3, 3), dtype=int)
for t, p in zip(true_regime, pred_regime):
    conf[t, p] += 1
row_ind, col_ind = linear_sum_assignment(-conf)
perm = dict(zip(col_ind, row_ind))  # fitted regime -> true label
aligned_pred = np.array([perm[p] for p in pred_regime])
accuracy = (aligned_pred == true_regime).mean()

print("confusion matrix (rows=true[low_low,high_low,high_high], cols=pred):\n", conf)
print("pred->true label mapping:", perm)
print("trial-level regime recovery accuracy (chance=1/3):", round(accuracy, 3))
for k in range(3):
    mask = true_regime == k
    print(
        f"  {TIER_NAMES[k]} trial-level accuracy: {(aligned_pred[mask] == k).mean():.3f} (n={mask.sum()})"
    )

block_acc = []
for b in np.unique(task.block_ids):
    mask = task.block_ids == b
    true_k = true_regime[mask][0]
    pred_k_mode = np.bincount(aligned_pred[mask]).argmax()
    block_acc.append(pred_k_mode == true_k)
print("per-block modal-regime accuracy:", np.mean(block_acc))

In [ ]:
inv_perm = {v: k for k, v in perm.items()}  # true label -> fitted regime
agent_of_true_regime = {
    0: 1,
    1: 2,
    2: 3,
}  # ground-truth: regime k's dominant agent is k+1

print(
    "param recovery (true label -> matched fitted regime -> that regime's dominant agent):"
)
for k in range(3):
    fk = inv_perm[k]
    fitted_betas = {a: model.params[f"beta_q{a}_{fk}"] for a in (1, 2, 3)}
    dominant_agent = max(fitted_betas, key=lambda a: abs(fitted_betas[a]))
    true_agent = agent_of_true_regime[k]
    print(
        f"  regime {k} ({TIER_NAMES[k]}) -> fitted regime {fk}, dominant agent {dominant_agent} "
        f"(betas={ {a: round(v, 2) for a, v in fitted_betas.items()} }):\n"
        f"    true agent {true_agent}: alpha_c={true_params[f'alpha_c_{true_agent}']}, "
        f"alpha_u={true_params[f'alpha_u_{true_agent}']}\n"
        f"    fit  agent {dominant_agent}: alpha_c={model.params[f'alpha_c_{dominant_agent}']:.3f}, "
        f"alpha_u={model.params[f'alpha_u_{dominant_agent}']:.3f}"
    )